# Financial DL Benchmark — Single Notebook (Audit + Pipeline)

Consolidated mirror of the runnable scripts; the `.py` files remain canonical and are kept.

- **Part A — Dataset audit** (mirror of `audit_dataset.py`): integrity check for git-ignored `data/global_indian_markets.csv`.
- **Part B — Full pipeline** (mirror of `run_benchmark.py`, 7 stages): configs, ingest, features/regimes, splits/scaling/windows, 4 model factories, OPTIONAL multi-seed training, metrics/regimes/costs/stats, figures.
- Kernel: `fin-dl-benchmark` (in-image venv — host untouched). Unexecuted, all outputs cleared.
- Training is gated by `RUN_TRAINING = False` — set `True` only for the expensive 4-model x 5-seed run.
- Replaces `02_run_benchmark.ipynb`, `03_dataset_audit.ipynb`, and the original `financial_time_series_benchmark.ipynb` (whose pre-Sept-2026-audit conclusions are intentionally NOT carried over; recoverable via git history). See `remaining_work.md`.


# Dataset Audit — Global & Indian Financial Markets

**Notebook mirror of `audit_dataset.py` (kept as the script version).**

- Purpose: lightweight integrity check — records, columns, dtypes, missing values, duplicates, SHA-256 hash.
- Input (git-ignored, not committed): `data/global_indian_markets.csv` from the Kaggle source in `instructions.txt`.
- This notebook is **unexecuted** (all outputs cleared). Run it inside the Docker Jupyter container so the host is untouched.
- Result is informational only — it does not train models or generate research results.
- See `remaining_work.md` Priority 0 for the full data-manifest next step.

In [ ]:
import hashlib
import pandas as pd

def audit_data(path="data/global_indian_markets.csv"):
    """Print a lightweight integrity report for the primary dataset."""
    if not pd.io.common.file_exists(path):
        print(f"Dataset not found at {path}. Please place the Kaggle CSV there.")
        return

    df = pd.read_csv(path)
    print("--- DATASET AUDIT REPORT ---")
    print(f"File Path: {path}")
    print(f"Total Records: {len(df):,}")
    print(f"Total Columns: {len(df.columns)}")
    print(f"Columns: {list(df.columns)}")
    print(f"Data Types:\n{df.dtypes}")
    print(f"Missing Values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print(f"Duplicate Rows: {df.duplicated().sum()}")

    with open(path, "rb") as f:
        file_hash = hashlib.sha256(f.read()).hexdigest()
    print(f"Dataset SHA-256 Hash: {file_hash}")


# To run: uncomment the next line inside Docker (data/ is mounted read-only).
# audit_data()

## Next steps

1. Place the exact Kaggle CSV at `data/global_indian_markets.csv` (see `DOCKER.md` mounts).
2. Run this cell set, then record file names, hashes, columns, coverage, and license in a data manifest (per `remaining_work.md`).
3. Do not commit the CSV — it is git-ignored.

# Financial DL Benchmark — Full Pipeline Runner (Notebook Mirror)

**Notebook mirror of `run_benchmark.py` (script version is kept and remains canonical).**

- Source: `run_benchmark.py` (366 lines, 7 stages). Code cells below are split by stage but preserve logic.
- Kernel: `fin-dl-benchmark` (created inside the Docker image from its in-image venv — host is untouched).
- This notebook is **unexecuted** (all outputs cleared) and **must not auto-train**: the multi-seed training cell is gated behind `RUN_TRAINING = False` and marked `OPTIONAL — EXECUTE TO TRAIN`.
- Known carryover (not fixed here): `configs/models.yaml` sets `mamba.allow_fallback: false` but the runner passes `allow_fallback=True`. Resolve before publishing any Mamba result (see `remaining_work.md`).
- Existing `results/` artifacts predate the Sept 2026 audit — retained for traceability, not report-ready (see `README.md`).

In [ ]:
import os
import sys
import yaml
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Ensure repo root is importable both on host checkout and in Docker (/home/jovyan/work)
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd())))
if os.path.exists("/home/jovyan/work/src"):
    sys.path.insert(0, "/home/jovyan/work")

from src.data.adapters import GlobalIndianAdapter
from src.data.features import build_features
from src.data.splits import chronological_split
from src.data.scaling import TimeSeriesScaler
from src.data.windows import create_rolling_windows
from src.regimes.rules import detect_regimes
from src.models.dlinear import DLinear
from src.models.lstm import LSTMSignalModel
from src.models.patchtst import PatchTSTSignalModel
from src.models.mamba_model import MambaSignalModel
from src.losses.sharpe_loss import NegativeSharpeLoss
from src.training.trainer import BaseTrainer
from src.backtest.backtester import Backtester
from src.evaluation.metrics import compute_all_metrics, calculate_sharpe_ratio
from src.evaluation.regime_metrics import evaluate_by_regime, calculate_regime_robustness
from src.evaluation.statistical_tests import block_bootstrap_sharpe, paired_sharpe_difference_test
from src.visualization.performance import (
    plot_market_overview,
    plot_regime_classification,
    plot_cumulative_returns,
    plot_drawdowns,
    plot_sharpe_by_regime,
    plot_turnover_vs_sharpe,
    plot_transaction_cost_sensitivity,
    plot_predictive_vs_economic,
    plot_complexity_tradeoff,
    plot_seed_robustness,
)

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def load_yaml(filepath: str) -> dict:
    with open(filepath, "r") as f:
        return yaml.safe_load(f)

# Safety switch: keep False for inspection; set True only to run the expensive benchmark.
RUN_TRAINING = False

## 1. Load configurations

Reads `configs/data.yaml`, `models.yaml`, `training.yaml`, `backtest.yaml`, `regime.yaml`. Creates `results/` dirs if missing.

In [ ]:
config_data = load_yaml("configs/data.yaml")
config_models = load_yaml("configs/models.yaml")
config_train = load_yaml("configs/training.yaml")
config_backtest = load_yaml("configs/backtest.yaml")
config_regime = load_yaml("configs/regime.yaml")

lookback = config_data.get("lookback", 20)
seeds = config_train.get("seeds", [42, 52, 62, 72, 82])
tc_bps_list = config_backtest.get("transaction_costs_bps", [0, 1, 5, 10, 20])
target_vol = config_backtest.get("target_volatility", 0.10)
max_lev = config_backtest.get("max_leverage", 3.0)

for output_dir in ("results/figures", "results/tables", "results/predictions"):
    os.makedirs(output_dir, exist_ok=True)

print(f"lookback={lookback}, seeds={seeds}, tc_bps={tc_bps_list}, target_vol={target_vol}, max_lev={max_lev}")

## 2. Data ingestion

Loads `data/global_indian_markets.csv` via `GlobalIndianAdapter` (git-ignored Kaggle CSV).

In [ ]:
adapter = GlobalIndianAdapter(data_path="data/global_indian_markets.csv")
prices = adapter.load()
print(f"Loaded dataset: {prices.shape[0]} daily records x {prices.shape[1]} assets: {list(prices.columns)}")
print(f"Date Range: {prices.index[0]} to {prices.index[-1]}")
plot_market_overview(prices, save_path="results/figures/01_market_data_overview.png")

## 3. Feature engineering & regime classification

Leakage-safe features via `build_features`; volatility x trend regimes via `detect_regimes` (thresholds should derive from train only — see `remaining_work.md`).

In [ ]:
feature_df, returns_df, vol_df, _ = build_features(prices)
regimes = detect_regimes(
    prices,
    vol_window=config_regime.get("volatility_window", 60),
    trend_window=config_regime.get("trend_window", 120),
)
plot_regime_classification(prices, regimes, save_path="results/figures/02_regime_classification.png")
print(regimes.value_counts().to_string())

## 4. Chronological splits, leakage-free scaling, rolling windows

60/20/20 chronological split; `TimeSeriesScaler` fit on train only; `DataLoader(shuffle=False)` preserves order for the batch Sharpe objective.

In [ ]:
train_feat_raw, val_feat_raw, test_feat_raw = chronological_split(feature_df, train_ratio=0.6, val_ratio=0.2)
train_ret_raw, val_ret_raw, test_ret_raw = chronological_split(returns_df, train_ratio=0.6, val_ratio=0.2)
train_vol_raw, val_vol_raw, test_vol_raw = chronological_split(vol_df, train_ratio=0.6, val_ratio=0.2)

scaler = TimeSeriesScaler()
scaler.fit(train_feat_raw.values)
train_feat_scaled = scaler.transform(train_feat_raw.values)
val_feat_scaled = scaler.transform(val_feat_raw.values)
test_feat_scaled = scaler.transform(test_feat_raw.values)

X_train, y_train, train_dates = create_rolling_windows(train_feat_scaled, train_ret_raw.values, lookback, train_feat_raw.index)
X_val, y_val, val_dates = create_rolling_windows(val_feat_scaled, val_ret_raw.values, lookback, val_feat_raw.index)
X_test, y_test, test_dates = create_rolling_windows(test_feat_scaled, test_ret_raw.values, lookback, test_feat_raw.index)

test_vol_aligned = test_vol_raw.iloc[lookback - 1: lookback - 1 + len(y_test)].values
test_regimes_aligned = regimes.loc[test_dates]

num_features = X_train.shape[-1]
num_assets = y_train.shape[-1]
print(f"Windowed shapes: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")

batch_size = config_train.get("batch_size", 64)
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32))
test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32))

# Chronological order preserved: shuffling would destroy local return structure for Sharpe loss.
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## 5. Model definitions (fair comparison)

Same data/features/lookback/splits/target/portfolio/costs for all four architectures. **Mamba warning:** config sets `allow_fallback: false` but the script factory below passes `allow_fallback=True` — do not publish fallback output as Mamba.

In [ ]:
model_factories = {
    "DLinear": lambda: DLinear(
        lookback=lookback,
        num_features=num_features,
        num_assets=num_assets,
        trend_window=config_models.get("dlinear", {}).get("trend_window", 5),
    ),
    "LSTM": lambda: LSTMSignalModel(
        lookback=lookback,
        num_features=num_features,
        num_assets=num_assets,
        hidden_dim=config_models.get("lstm", {}).get("hidden_dim", 64),
        num_layers=config_models.get("lstm", {}).get("layers", 2),
        dropout=config_models.get("lstm", {}).get("dropout", 0.1),
    ),
    # NOTE: allow_fallback=True here contradicts configs/models.yaml (false). Kept for fidelity;
    # set to False / install mamba_ssm before any Mamba claim.
    "Mamba": lambda: MambaSignalModel(
        lookback=lookback,
        num_features=num_features,
        num_assets=num_assets,
        d_model=config_models.get("mamba", {}).get("d_model", 64),
        n_layers=config_models.get("mamba", {}).get("layers", 2),
        allow_fallback=True,
    ),
    "PatchTST": lambda: PatchTSTSignalModel(
        lookback=lookback,
        num_features=num_features,
        num_assets=num_assets,
        patch_length=config_models.get("patchtst", {}).get("patch_length", 5),
        d_model=config_models.get("patchtst", {}).get("d_model", 64),
        n_heads=config_models.get("patchtst", {}).get("heads", 4),
        n_layers=config_models.get("patchtst", {}).get("layers", 2),
        dropout=0.1,
    ),
}

for name, factory in model_factories.items():
    print(f"{name}: {factory().count_parameters():,} parameters")

## 6. Training & multi-seed benchmarking — OPTIONAL — EXECUTE TO TRAIN

Expensive: 4 architectures x 5 seeds. Only runs when `RUN_TRAINING = True`. Includes equal-weight buy-and-hold baseline at 5 bps.

In [ ]:
if not RUN_TRAINING:
    print("RUN_TRAINING=False — skipping expensive training (inspection mode). Set True to execute.")
else:
    model_params = {}
    model_train_times = {m: [] for m in model_factories.keys()}
    seed_sharpe_results = {m: [] for m in model_factories.keys()}
    primary_test_returns = {}
    primary_test_turnover = {}
    primary_test_signals = {}

    ew_positions = np.ones_like(y_test) * (1.0 / num_assets)
    ew_backtester = Backtester(target_vol=target_vol, max_leverage=max_lev, tc_bps=5.0)
    ew_res = ew_backtester.run(ew_positions, test_vol_aligned, y_test, test_dates)
    primary_test_returns["Buy & Hold (EW)"] = ew_res["net_returns"]
    primary_test_turnover["Buy & Hold (EW)"] = 0.0

    for model_name, factory in model_factories.items():
        sample_model = factory()
        model_params[model_name] = sample_model.count_parameters()
        print(f"--- {model_name} ({model_params[model_name]:,} params) ---")
        for seed_idx, seed in enumerate(seeds):
            set_seed(seed)
            model = factory()
            criterion = NegativeSharpeLoss(eps=1e-6, ann_factor=252.0, alpha=config_train.get("loss_alpha", 1.0))
            optimizer = optim.AdamW(model.parameters(), lr=config_train.get("learning_rate", 0.001), weight_decay=1e-4)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config_train.get("epochs", 50))
            trainer = BaseTrainer(model, criterion, optimizer, config_train, scheduler=scheduler)
            history, duration = trainer.train(train_loader, val_loader)
            model_train_times[model_name].append(duration)
            signals = trainer.predict(test_loader)
            backtester = Backtester(target_vol=target_vol, max_leverage=max_lev, tc_bps=5.0)
            bt_res = backtester.run(signals, test_vol_aligned, y_test, test_dates)
            net_ret = bt_res["net_returns"]
            net_sharpe = calculate_sharpe_ratio(net_ret)
            seed_sharpe_results[model_name].append(net_sharpe)
            if seed_idx == 0:
                primary_test_returns[model_name] = net_ret
                primary_test_turnover[model_name] = float(np.nanmean(bt_res["turnover"]) * 252.0)
                primary_test_signals[model_name] = signals
            print(f"  Seed {seed} -> val_loss={history['val_loss'][-1]:.4f}, net Sharpe={net_sharpe:.3f} ({duration:.1f}s)")

## 7. Metrics, regimes, costs, statistics (runs after training)

Writes `results/tables/01..05.*`. Requires the `RUN_TRAINING` outputs above.

In [ ]:
if not RUN_TRAINING:
    print("Skipped — requires training outputs. Set RUN_TRAINING=True first.")
else:
    metrics_records = []
    for name, ret_series in primary_test_returns.items():
        m_dict = compute_all_metrics(ret_series)
        point_s, ci_low, ci_high = block_bootstrap_sharpe(ret_series.values, block_size=20, num_bootstraps=1000)
        m_dict["Model"] = name
        m_dict["95% Sharpe CI"] = f"[{ci_low:.2f}, {ci_high:.2f}]"
        m_dict["Turnover (ann.)"] = f"{primary_test_turnover.get(name, 0.0):.2f}"
        if name in model_params:
            m_dict["Parameters"] = f"{model_params[name]:,}"
            m_dict["Mean Train Time (s)"] = f"{np.mean(model_train_times[name]):.1f}"
            m_dict["Seed Sharpe Mean \u00b1 Std"] = f"{np.mean(seed_sharpe_results[name]):.2f} \u00b1 {np.std(seed_sharpe_results[name]):.2f}"
        else:
            m_dict["Parameters"] = "N/A"
            m_dict["Mean Train Time (s)"] = "N/A"
            m_dict["Seed Sharpe Mean \u00b1 Std"] = "N/A"
        metrics_records.append(m_dict)
    df_metrics = pd.DataFrame(metrics_records)
    df_metrics.to_csv("results/tables/01_primary_economic_metrics.csv", index=False)
    with open("results/tables/01_primary_economic_metrics.md", "w") as f:
        f.write("# Primary Economic Performance Summary\n\n")
        f.write(df_metrics.to_markdown(index=False))
    print(df_metrics.to_string(index=False))

    regime_results = {}
    regime_robustness_records = []
    for model_name in model_factories.keys():
        ret_s = primary_test_returns[model_name]
        reg_eval = evaluate_by_regime(ret_s, test_regimes_aligned)
        regime_results[model_name] = reg_eval["sharpe"]
        rob_dict = calculate_regime_robustness(ret_s, test_regimes_aligned)
        rob_dict["Model"] = model_name
        regime_robustness_records.append(rob_dict)
    df_regime_sharpes = pd.DataFrame(regime_results).T.fillna(0.0)
    df_regime_sharpes.to_csv("results/tables/02_regime_sharpes.csv")
    df_robustness = pd.DataFrame(regime_robustness_records).set_index("Model")
    df_robustness.to_csv("results/tables/03_regime_robustness_scores.csv")

    tc_sensitivity_data = {}
    for model_name, sigs in primary_test_signals.items():
        tc_sensitivity_data[model_name] = {}
        for bps in tc_bps_list:
            bt = Backtester(target_vol=target_vol, max_leverage=max_lev, tc_bps=bps)
            r = bt.run(sigs, test_vol_aligned, y_test, test_dates)["net_returns"]
            tc_sensitivity_data[model_name][bps] = calculate_sharpe_ratio(r)
    df_tc = pd.DataFrame(tc_sensitivity_data).T
    df_tc.columns = [f"{b} bps" for b in df_tc.columns]
    df_tc.to_csv("results/tables/04_transaction_cost_sensitivity.csv")

    pred_acc_dict = {}
    for model_name, sigs in primary_test_signals.items():
        pred_acc_dict[model_name] = float(np.mean((np.sign(sigs) == np.sign(y_test))))

    stat_test_records = []
    model_names_list = list(model_factories.keys())
    for i in range(len(model_names_list)):
        for j in range(i + 1, len(model_names_list)):
            m_a, m_b = model_names_list[i], model_names_list[j]
            test_res = paired_sharpe_difference_test(primary_test_returns[m_a].values, primary_test_returns[m_b].values, num_bootstraps=1000)
            stat_test_records.append({"Comparison": f"{m_a} vs {m_b}", "Sharpe (A)": f"{test_res['sharpe_a']:.3f}", "Sharpe (B)": f"{test_res['sharpe_b']:.3f}", "Difference (A - B)": f"{test_res['sharpe_diff']:.3f}", "Bootstrap p-value": f"{test_res['p_value']:.4f}"})
    df_stats = pd.DataFrame(stat_test_records)
    df_stats.to_csv("results/tables/05_hypothesis_tests.csv", index=False)

## 8. Publication figures (runs after training)

Writes `results/figures/03..10.png` at report quality.

In [ ]:
if not RUN_TRAINING:
    print("Skipped — requires training outputs.")
else:
    plot_cumulative_returns(primary_test_returns, save_path="results/figures/03_cumulative_returns.png")
    plot_drawdowns(primary_test_returns, save_path="results/figures/04_drawdown_curves.png")
    plot_sharpe_by_regime(df_regime_sharpes, save_path="results/figures/05_sharpe_by_regime.png")
    primary_sharpes = {m: df_metrics.set_index("Model").loc[m, "Sharpe Ratio"] for m in model_factories.keys()}
    plot_turnover_vs_sharpe(primary_test_turnover, primary_sharpes, save_path="results/figures/06_turnover_vs_sharpe.png")
    plot_transaction_cost_sensitivity(tc_sensitivity_data, save_path="results/figures/07_transaction_cost_sensitivity.png")
    plot_predictive_vs_economic(pred_acc_dict, primary_sharpes, save_path="results/figures/08_predictive_vs_economic.png")
    plot_complexity_tradeoff(model_params, primary_sharpes, save_path="results/figures/09_complexity_tradeoff.png")
    plot_seed_robustness(seed_sharpe_results, save_path="results/figures/10_seed_robustness.png")
    print("Figures written to results/figures/")

## Reproducibility checklist

- `src/` stays importable (no notebook-only logic); this notebook only orchestrates.
- Seeds `[42, 52, 62, 72, 82]` from `configs/training.yaml`; chronological splits; train-only scaler; `shuffle=False`.
- Before any Mamba claim: install `mamba_ssm` and set `allow_fallback: false` end-to-end.
- Regenerate `results/` from the verified Kaggle CSV — current committed tables/figures predate the audit.

## Test-suite mirrors (inspection only)

Mirrors of `tests/*.py` for reading alongside the pipeline. The `.py` files remain canonical — run them with pytest, not by executing these cells.

```bash
docker compose run --rm jupyter /opt/venv/bin/python -m pytest tests/ -v
```

All cells below are unexecuted (`execution_count: null`, no outputs).


### `tests/test_data_integrity.py`

Leakage/alignment guards: rolling volatility warm-up is NaN (no backfill), feature/return/vol indexes stay aligned, rolling windows keep next-return alignment, and an unavailable Mamba dependency raises instead of silently becoming an LSTM.


In [ ]:
import numpy as np
import pandas as pd
import pytest

from src.data.features import build_features, calculate_rolling_volatility
from src.data.windows import create_rolling_windows
from src.models.mamba_model import MAMBA_AVAILABLE, MambaSignalModel


def test_rolling_volatility_does_not_backfill_warmup_period():
    dates = pd.date_range("2020-01-01", periods=30, freq="D")
    returns = pd.DataFrame({"asset": np.linspace(-0.01, 0.01, 30)}, index=dates)

    volatility = calculate_rolling_volatility(returns, window=20)

    assert volatility.iloc[:19].isna().all().all()
    assert volatility.iloc[19:].notna().all().all()


def test_feature_outputs_are_aligned_after_warmup_removal():
    dates = pd.date_range("2020-01-01", periods=100, freq="D")
    prices = pd.DataFrame(
        {
            "asset_a": 100.0 * np.cumprod(1.0 + np.linspace(-0.002, 0.003, 100)),
            "asset_b": 80.0 * np.cumprod(1.0 + np.linspace(0.001, -0.001, 100)),
        },
        index=dates,
    )

    features, returns, vol_20, feature_dict = build_features(prices)

    assert features.index.equals(returns.index)
    assert features.index.equals(vol_20.index)
    assert not features.isna().any().any()
    assert features.index[0] == dates[59]
    assert all(frame.index.equals(features.index) for frame in feature_dict.values())


def test_rolling_windows_keep_next_return_alignment():
    dates = pd.date_range("2020-01-01", periods=6, freq="D")
    features = np.arange(12, dtype=float).reshape(6, 2)
    targets = np.arange(6, dtype=float).reshape(6, 1)

    x, y, decision_dates = create_rolling_windows(features, targets, lookback=3, dates=dates)

    np.testing.assert_array_equal(x[0], features[:3])
    np.testing.assert_array_equal(y[:, 0], targets[3:, 0])
    assert decision_dates[0] == dates[2]


def test_mamba_cannot_silently_become_lstm():
    if MAMBA_AVAILABLE:
        pytest.skip("The real mamba_ssm dependency is installed.")

    with pytest.raises(ImportError, match="mamba_ssm is required"):
        MambaSignalModel(lookback=20, num_features=4, num_assets=2)


### `tests/test_shapes.py`

Forward-pass shape contract for all four architectures: `[batch, lookback, features]` in, `[batch, assets]` bounded signal out. The Mamba fallback is permitted here only for lightweight interface validation.


In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))
import torch
from src.models.dlinear import DLinear
from src.models.lstm import LSTMSignalModel
from src.models.patchtst import PatchTSTSignalModel
from src.models.mamba_model import MambaSignalModel

def test_model_shapes():
    batch_size = 32
    lookback = 20
    num_assets = 10
    num_features = num_assets # Simplified for this test
    
    x = torch.randn(batch_size, lookback, num_features)
    
    # 1. DLinear
    model_dlinear = DLinear(lookback=lookback, num_assets=num_assets)
    out_dlinear = model_dlinear(x)
    assert out_dlinear.shape == (batch_size, num_assets), f"DLinear shape error: {out_dlinear.shape}"
    
    # 2. LSTM
    model_lstm = LSTMSignalModel(lookback=lookback, num_features=num_features, num_assets=num_assets)
    out_lstm = model_lstm(x)
    assert out_lstm.shape == (batch_size, num_assets), f"LSTM shape error: {out_lstm.shape}"
    
    # 3. PatchTST
    # Note: PatchTST expects lookback divisible by patch_length
    model_patch = PatchTSTSignalModel(lookback=lookback, num_features=num_features, num_assets=num_assets, patch_length=5)
    out_patch = model_patch(x)
    assert out_patch.shape == (batch_size, num_assets), f"PatchTST shape error: {out_patch.shape}"
    
    # 4. Mamba (or fallback)
    # The fallback is permitted only for lightweight interface validation.
    model_mamba = MambaSignalModel(
        lookback=lookback,
        num_features=num_features,
        num_assets=num_assets,
        allow_fallback=True,
    )
    out_mamba = model_mamba(x)
    assert out_mamba.shape == (batch_size, num_assets), f"Mamba shape error: {out_mamba.shape}"
    
    print("All model shapes validated successfully.")

if __name__ == "__main__":
    test_model_shapes()


### `tests/test_optimizations.py`

Equivalence guards for the optimized paths: vectorized rolling windows, reused turnover/transaction-cost math vs the backtester, and the vectorized block bootstrap vs the legacy reference.


In [ ]:
import numpy as np
import pandas as pd

from src.backtest.backtester import Backtester
from src.data.windows import create_rolling_windows
from src.evaluation.statistical_tests import block_bootstrap_sharpe
from src.portfolio.transaction_costs import (
    calculate_position_turnover,
    calculate_transaction_costs,
)


def _legacy_windows(features, targets, lookback, dates=None):
    x_values, y_values, valid_dates = [], [], []
    for index in range(len(features) - lookback):
        x_values.append(features[index : index + lookback])
        y_values.append(targets[index + lookback])
        if dates is not None:
            valid_dates.append(dates[index + lookback - 1])
    return (
        np.array(x_values, dtype=np.float32),
        np.array(y_values, dtype=np.float32),
        pd.DatetimeIndex(valid_dates) if dates is not None else None,
    )


def _legacy_block_bootstrap(
    returns, block_size=20, num_bootstraps=1000, random_seed=42
):
    np.random.seed(random_seed)
    ret_arr = np.asarray(returns)
    ret_arr = ret_arr[~np.isnan(ret_arr)]
    sample_count = len(ret_arr)
    point_mean = np.mean(ret_arr) * 252.0
    point_vol = np.std(ret_arr, ddof=1) * np.sqrt(252.0)
    point_sharpe = float(point_mean / point_vol) if point_vol > 1e-8 else 0.0
    block_count = max(1, sample_count // block_size)
    sharpes = []

    for _ in range(num_bootstraps):
        starts = np.random.randint(
            0, sample_count - block_size + 1, size=block_count
        )
        boot_returns = []
        for start in starts:
            boot_returns.extend(ret_arr[start : start + block_size])
        boot_arr = np.array(boot_returns)
        mean_return = np.mean(boot_arr) * 252.0
        volatility = np.std(boot_arr, ddof=1) * np.sqrt(252.0)
        if volatility > 1e-8:
            sharpes.append(mean_return / volatility)

    return (
        point_sharpe,
        float(np.percentile(sharpes, 2.5)),
        float(np.percentile(sharpes, 97.5)),
    )


def test_vectorized_windows_match_legacy_output_exactly():
    generator = np.random.default_rng(7)
    features = generator.normal(size=(40, 6))
    targets = generator.normal(size=(40, 3))
    dates = pd.date_range("2021-01-01", periods=40, freq="D")

    expected = _legacy_windows(features, targets, lookback=8, dates=dates)
    actual = create_rolling_windows(features, targets, lookback=8, dates=dates)

    np.testing.assert_array_equal(actual[0], expected[0])
    np.testing.assert_array_equal(actual[1], expected[1])
    assert actual[2].equals(expected[2])
    assert actual[0].flags.c_contiguous


def test_reused_turnover_matches_previous_backtest_calculation():
    positions = np.array(
        [[0.5, -0.25], [0.75, -0.10], [-0.20, 0.30]], dtype=float
    )
    shifted = np.roll(positions, shift=1, axis=0)
    shifted[0] = 0.0
    expected_turnover = np.abs(positions - shifted)

    turnover = calculate_position_turnover(positions)
    costs = calculate_transaction_costs(positions, bps=5.0, turnover=turnover)

    np.testing.assert_array_equal(turnover, expected_turnover)
    np.testing.assert_array_equal(costs, expected_turnover * 0.0005)

    signals = positions
    volatility = np.full_like(signals, 0.10)
    returns = np.full_like(signals, 0.01)
    result = Backtester(target_vol=0.10, max_leverage=3.0, tc_bps=5.0).run(
        signals, volatility, returns
    )
    np.testing.assert_array_equal(
        result["turnover"].to_numpy(), expected_turnover.mean(axis=1)
    )


def test_vectorized_bootstrap_matches_legacy_result_exactly():
    returns = np.linspace(-0.02, 0.025, 120)
    expected = _legacy_block_bootstrap(
        returns, block_size=12, num_bootstraps=50, random_seed=19
    )
    actual = block_bootstrap_sharpe(
        returns, block_size=12, num_bootstraps=50, random_seed=19
    )

    np.testing.assert_allclose(actual, expected, rtol=0.0, atol=0.0)


## What to run, and where

- Inspect here; execute in Docker: `docker compose run --rm jupyter /opt/venv/bin/python -m pytest tests/ -v`.
- Do not treat these synthetic checks as research results (see `remaining_work.md` Priority 3).
